In [ ]:
import pandas as pd

from graphdb_connector import connector

In [ ]:
with open('assets/experiments.xlsx', 'rb') as f:
    proteomics_data = pd.read_excel(f, sheet_name="Proteomics", header=1)
    phosphoSTY_data = pd.read_excel(f, sheet_name="Phospho-pSTY", header=1)
    phosphoY_data = pd.read_excel(f, sheet_name="Phospho-pY", header=1)
    acetylomics_data = pd.read_excel(f, sheet_name="Acetylomics-acK", header=1)
    rnaseq_data = pd.read_excel(f, sheet_name="RNA-seq", header=1)
with open('assets/samples.xlsx', 'rb') as f:
    clinical_data = pd.read_excel(f, header=0)

In [ ]:
proteomics_data.head()

In [ ]:
phosphoSTY_data.head()

In [ ]:
accessions = set(proteomics_data['accession_number'].values.tolist())
accessions.update(phosphoSTY_data['accession_number'].values.tolist())
accessions.update(phosphoY_data['accession_number'].values.tolist())
accessions.update(acetylomics_data['accession_number'].values.tolist())

In [ ]:
mapping_query = 'MATCH (t:Transcript)-[:TRANSLATED_INTO]-(p:Protein) WHERE (t.id IN [{}] OR p.name IN [{}]) AND NOT p.id=~".*-.*" RETURN t.id, p.id ORDER BY t.id,p.id'

In [ ]:
driver = connector.getGraphDatabaseConnectionConfiguration()

In [ ]:
identifiers = ['"{}"'.format(i) for i in list(accessions)]
names = ['"{}"'.format(g) for g in proteomics_data['geneSymbol'].values.tolist()]

In [ ]:
mapping = connector.getCursorData(driver, mapping_query.format(",".join(identifiers), ",".join(names)))
mapping = mapping.drop_duplicates(subset="t.id", keep='first')

In [ ]:
mapping = mapping.set_index("t.id").to_dict(orient='dict')["p.id"]

In [ ]:
proteomics_data["identifier"] = [mapping[p] if p in mapping else p for p in proteomics_data["accession_number"].tolist()]
phosphoSTY_data["identifier"] = [mapping[p] if p in mapping else p for p in phosphoSTY_data["accession_number"].tolist()]
phosphoY_data["identifier"] = [mapping[p] if p in mapping else p for p in phosphoY_data["accession_number"].tolist()]
acetylomics_data["identifier"] = [mapping[p] if p in mapping else p for p in acetylomics_data["accession_number"].tolist()]

In [ ]:
proteomics_data[proteomics_data['identifier'] == 'P01106']

In [ ]:
phosphoSTY_data["variableSites"] = [m.split(' ')[0][0:-1]+"-p" for m in phosphoSTY_data["variableSites"].tolist()]
phosphoY_data["variableSites"] = [m.split(' ')[0][0:-1]+"-p" for m in phosphoY_data["variableSites"].tolist()]
acetylomics_data["variableSites"] = [m.split(' ')[0][0:-1]+"-ac" for m in acetylomics_data["variableSites"].tolist()]

In [ ]:
phosphoSTY_data["identifier"] = phosphoSTY_data["identifier"] +"_"+phosphoSTY_data["variableSites"]
phosphoY_data["identifier"] = phosphoY_data["identifier"] +"_"+phosphoY_data["variableSites"]
acetylomics_data["identifier"] = acetylomics_data["identifier"] +"_"+acetylomics_data["variableSites"]

In [ ]:
rnaseq_data = rnaseq_data.set_index("Transcript / Genesym")
rnaseq_data.index.name = "identifier"
rnaseq_data = rnaseq_data.loc[:,rnaseq_data.columns.isin(clinical_data['sample'].tolist())].transpose().join(clinical_data.set_index('sample')[['group']]).reset_index()
proteomics_data = proteomics_data.set_index("identifier")[clinical_data['sample'].tolist()].transpose().join(clinical_data.set_index('sample')[['group']]).reset_index()
phosphoSTY_data = phosphoSTY_data.set_index("identifier")[clinical_data['sample'].tolist()].transpose().join(clinical_data.set_index('sample')[['group']]).reset_index()
phosphoY_data = phosphoY_data.set_index("identifier")[clinical_data['sample'].tolist()].transpose().join(clinical_data.set_index('sample')[['group']]).reset_index()
acetylomics_data = acetylomics_data.set_index("identifier")[clinical_data['sample'].tolist()].transpose().join(clinical_data.set_index('sample')[['group']]).reset_index()

In [ ]:
proteomics_data.to_csv(path_or_buf="assets/proteomics.tsv", sep='\t', header=True, index=False, quotechar='"', line_terminator='\n', escapechar='\\')
phosphoSTY_data.to_csv(path_or_buf="assets/phosphoSTY.tsv", sep='\t', header=True, index=False, quotechar='"', line_terminator='\n', escapechar='\\')
phosphoY_data.to_csv(path_or_buf="assets/phosphoY.tsv", sep='\t', header=True, index=False, quotechar='"', line_terminator='\n', escapechar='\\')
acetylomics_data.to_csv(path_or_buf="assets/acetylomics.tsv", sep='\t', header=True, index=False, quotechar='"', line_terminator='\n', escapechar='\\')
rnaseq_data.to_csv(path_or_buf="assets/rnaseq.tsv", sep='\t', header=True, index=False, quotechar='"', line_terminator='\n', escapechar='\\')

In [ ]:
proteomics_data.head()